In [1]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.interpolate import BPoly

In [2]:
# ============================================================
# -- Switch  (only change this one line) ---------------------
# ============================================================

CASE = 2   # 1 = Quadratic 2D,  2 = Quartic 3D


# ============================================================
# -- CASE Data  (do not change) --------------------------
# ============================================================

CASE_DATA = {
    1: dict(
        points = np.array([
            [1, 1],   # P0
            [3, 3],   # P1
            [5, 1],   # P2
        ]),
        u_highlight = [0.00, 0.25, 0.50, 0.75, 1.00],
        u_dc        = 0.50,
        output      = '35-bezier_curvature_quadratic_2d.png',
    ),
    2: dict(
        points = np.array([
            [0, 0, 1],   # P0
            [0, 4, 1],   # P1
            [4, 0, 1],   # P2
            [4, 4, 1],   # P3
            [5, 4, 1],   # P4
        ]),
        u_highlight = [0.00, 0.25, 0.50, 0.75, 1.00],
        u_dc        = 0.50,
        output      = '35-bezier_curvature_quartic_3d.png',
    ),
}

In [3]:
# ============================================================
# -- BezierCurve  (generic — do not change) ------------------
# ============================================================

class BezierCurve:
    """
    Generic Bezier curve of any degree, 2D or 3D.

    Curve evaluation uses scipy.interpolate.BPoly.

    Curvature uses derivative Bezier curves:
      P'(u)  — BPoly with control points Q_i = n*(P_{i+1} - P_i)
      P''(u) — BPoly with control points R_i = (n-1)*(Q_{i+1} - Q_i)

    kappa = |P' x P''| / |P'|^3
      2D: cross product is scalar  x'y'' - y'x''
      3D: cross product is a vector, take its norm
    """

    def __init__(self, points: np.ndarray):
        self.points = points
        self.n      = len(points)
        self.degree = self.n - 1
        self.dim    = points.shape[1]

        c = points[:, np.newaxis, :]
        self._bpoly = BPoly(c, [0, 1])

        # ── Derivative BPolys ────────────────────────────
        Q = self.degree * np.diff(points, axis=0)
        self._bpoly_d1 = BPoly(Q[:, np.newaxis, :], [0, 1])

        if len(Q) > 1:
            R = (self.degree - 1) * np.diff(Q, axis=0)
            self._bpoly_d2 = BPoly(R[:, np.newaxis, :], [0, 1])
        else:
            self._bpoly_d2 = None

    def point(self, u: float) -> np.ndarray:
        return self._bpoly(u)

    def curve(self, n_pts: int = 300) -> np.ndarray:
        return self._bpoly(np.linspace(0, 1, n_pts))

    def highlight_points(self, u_list: list) -> np.ndarray:
        return self._bpoly(np.array(u_list))

    def de_casteljau(self, u: float) -> list:
        levels  = [self.points.copy()]
        current = self.points.copy()
        while len(current) > 1:
            current = np.array([
                (1 - u) * current[j] + u * current[j + 1]
                for j in range(len(current) - 1)
            ])
            levels.append(current)
        return levels

    def d1(self, u):
        """First derivative P'(u)."""
        return self._bpoly_d1(u)

    def d2(self, u):
        """Second derivative P''(u)."""
        if self._bpoly_d2 is None:
            return np.zeros((len(np.atleast_1d(u)), self.dim))
        return self._bpoly_d2(u)

    def curvature(self, u):
        """kappa(u) at one or many u values."""
        u_arr = np.atleast_1d(u)
        d1    = self.d1(u_arr)
        d2    = self.d2(u_arr)
        if self.dim == 2:
            cross = np.abs(d1[:, 0]*d2[:, 1] - d1[:, 1]*d2[:, 0])
        else:
            cross = np.linalg.norm(np.cross(d1, d2), axis=1)
        speed = np.linalg.norm(d1, axis=1)
        denom = np.where(speed > 1e-10, speed**3, np.inf)
        return cross / denom

    def comb_scale(self, n_pts=300):
        """Auto-scale: comb height ~ 15% of curve extent."""
        curve_pts = self.curve(n_pts)
        u_vals    = np.linspace(0, 1, n_pts)
        kappa     = self.curvature(u_vals)
        ext       = np.ptp(curve_pts[:, :2], axis=0).max()
        kap_max   = np.percentile(kappa, 95)
        return (0.15 * ext / kap_max) if kap_max > 1e-10 else 0.1

    @property
    def degree_name(self) -> str:
        names = {2: 'Quadratic', 3: 'Cubic', 4: 'Quartic', 5: 'Quintic'}
        return names.get(self.degree, f'Degree-{self.degree}')

    @property
    def dim_name(self) -> str:
        return '3D' if self.dim == 3 else '2D'


In [4]:
# ============================================================
# -- Plot  (generic — do not change) -------------------------
# ============================================================

class BezierPlot:
    """
    Plot a Bezier curve in 2D or 3D automatically.

    Detects dimensionality from bezier.dim and sets up
    the correct axes (standard or 3D projection).
    All drawing logic is shared — only axis calls differ.
    """

    COLOR_CURVE   = '#1565C0'
    COLOR_POLYGON = '#E53935'
    COLOR_CTRL    = '#E53935'
    COLOR_POINT   = '#1B5E20'
    COLOR_LABEL   = '#4A148C'
    COLOR_DC = [
        '#F57F17',   # level 1 — amber
        '#EC407A',   # level 2 — pink
        '#7B1FA2',   # level 3 — violet
        '#1B5E20',   # level 4 — green (final point)
    ]

    def __init__(
        self,
        bezier:      BezierCurve,
        u_highlight: list,
        u_dc:        float = 0.5,
    ):
        self.bezier      = bezier
        self.u_highlight = u_highlight
        self.u_dc        = u_dc
        self.is_3d       = (bezier.dim == 3)
        self.fig, self.ax = self._init_figure()

    # -- Figure setup ----------------------------------------

    def _init_figure(self):
        sns.set_theme(style='whitegrid')
        if self.is_3d:
            fig = plt.figure(figsize=(10, 7))
            ax  = fig.add_subplot(111, projection='3d')
        else:
            fig, ax = plt.subplots(figsize=(9, 7))
        return fig, ax

    # -- Unified plot helpers (2D / 3D) ----------------------

    def _coords(self, pts: np.ndarray) -> tuple:
        """
        Unpack point array into coordinate tuples.
        Returns (x, y) for 2D or (x, y, z) for 3D.
        """
        if self.is_3d:
            return pts[:, 0], pts[:, 1], pts[:, 2]
        return pts[:, 0], pts[:, 1]

    def _plot_line(self, pts, **kwargs):
        """Draw a line through pts in either 2D or 3D."""
        self.ax.plot(*self._coords(pts), **kwargs)

    def _scatter_pts(self, pts, **kwargs):
        """Scatter-plot pts in either 2D or 3D."""
        if self.is_3d:
            xs, ys, zs = self._coords(pts)
            self.ax.scatter(xs, ys, zs, **kwargs)
        else:
            xs, ys = self._coords(pts)
            self.ax.scatter(xs, ys, **kwargs)

    def _annotate(self, pt, text, dx, dy, dz, color):
        """Place a text label near a point in 2D or 3D."""
        if self.is_3d:
            self.ax.text(
                pt[0] + dx, pt[1] + dy, pt[2] + dz,
                text, fontsize=8, color=color,
                fontweight='bold',
            )
        else:
            self.ax.annotate(
                text,
                xy     = (pt[0], pt[1]),
                xytext = (pt[0] + dx, pt[1] + dy),
                fontsize   = 8.5,
                color      = color,
                fontweight = 'bold',
            )

    # -- Drawing methods -------------------------------------

    def _draw_curve(self):
        """Draw the smooth Bezier curve."""
        pts = self.bezier.curve()
        self._plot_line(
            pts,
            color     = self.COLOR_CURVE,
            linewidth = 2.5,
            label     = 'Bezier Curve',
            zorder    = 3,
        )

    def _draw_control_polygon(self):
        """Draw dashed polygon connecting all control points."""
        self._plot_line(
            self.bezier.points,
            color      = self.COLOR_POLYGON,
            linestyle  = '--',
            linewidth  = 1.5,
            marker     = 'o',
            markersize = 8,
            label      = 'Control Polygon',
            zorder     = 4,
        )

    def _label_control_points(self):
        """Label each control point as P0, P1, ..."""
        for i, pt in enumerate(self.bezier.points):
            coords = ', '.join(f'{v:.4g}' for v in pt)
            name   = f'P{i}({coords})'
            dy     = 0.15 if i % 2 == 0 else -0.28
            self._annotate(
                pt, name,
                dx=0.10, dy=dy, dz=0.05,
                color=self.COLOR_CTRL,
            )

    def _draw_highlight_points(self):
        """Draw and annotate curve points at u_highlight."""
        pts = self.bezier.highlight_points(self.u_highlight)
        self._scatter_pts(
            pts,
            color  = self.COLOR_POINT,
            s      = 60,
            zorder = 5,
            label  = 'Points at u',
        )
        for u, pt in zip(self.u_highlight, pts):
            coords = ', '.join(f'{v:.3g}' for v in pt)
            dx = -0.55 if u == 1.0 else 0.08
            dy = -0.28 if u in (0.0, 1.0) else 0.10
            if self.is_3d:
                self.ax.text(
                    pt[0] + 0.08, pt[1] + 0.08, pt[2] + 0.05,
                    f'u={u:.2f}',
                    fontsize=7, color=self.COLOR_LABEL,
                )
            else:
                self.ax.annotate(
                    f'u={u:.2f}\n({coords})',
                    xy     = (pt[0], pt[1]),
                    xytext = (pt[0] + dx, pt[1] + dy),
                    fontsize = 7.5,
                    color    = self.COLOR_LABEL,
                    bbox     = dict(
                        boxstyle='round,pad=0.2',
                        fc='white', alpha=0.65, ec='none',
                    ),
                )

    def _draw_de_casteljau(self):
        """
        Draw De Casteljau intermediate levels at u_dc.
        Works identically for 2D and 3D.
        """
        levels = self.bezier.de_casteljau(self.u_dc)

        for lv in range(1, len(levels) - 1):
            pts = levels[lv]
            col = self.COLOR_DC[
                min(lv - 1, len(self.COLOR_DC) - 1)
            ]
            self._plot_line(
                pts,
                color      = col,
                linewidth  = 1.4,
                marker     = 'o',
                markersize = 6,
                zorder     = 6,
                label      = f'DC level {lv}',
            )

        final = levels[-1]
        col   = self.COLOR_DC[
            min(len(levels) - 2, len(self.COLOR_DC) - 1)
        ]
        self._scatter_pts(
            final,
            color  = col,
            marker = 'D',
            s      = 90,
            zorder = 7,
            label  = f'DC final  u={self.u_dc:.2f}',
        )

    def _add_equation_box(self):
        """
        Show what BPoly computes — the Bernstein weighted sum.
        The formula is the same; scipy just evaluates it internally.
        """
        n     = self.bezier.degree
        terms = ' + '.join(
            f'B({n},{i})\u00b7P{i}'
            for i in range(self.bezier.n)
        )
        text = (
            f'P(u) = {terms}\n'
            f'B(n,i,u) = C(n,i)\u00b7u\u1d35\u00b7(1-u)^(n-i)\n'
            f'[evaluated via scipy.interpolate.BPoly]'
        )
        if self.is_3d:
            self.fig.text(
                0.13, 0.92, text,
                fontsize=8, fontfamily='monospace', va='top',
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='lightyellow', alpha=0.85),
            )
        else:
            self.ax.text(
                0.02, 0.97, text,
                transform=self.ax.transAxes,
                fontsize=8, fontfamily='monospace', va='top',
                bbox=dict(boxstyle='round,pad=0.4',
                          facecolor='lightyellow', alpha=0.85),
            )


    def _draw_comb(self):
        """
        Curvature comb — perpendicular spikes, length ~ kappa.
        2D: normal = tangent rotated 90 degrees.
        3D: principal normal N = (P' x P'') x P'.
        """
        n_pts  = 300
        u_vals = np.linspace(0, 1, n_pts)
        scale  = self.bezier.comb_scale(n_pts)
        step   = max(1, n_pts // 60)
        u_sub  = u_vals[::step]

        d1v    = self.bezier.d1(u_sub)
        d2v    = self.bezier.d2(u_sub)
        base   = self.bezier._bpoly(u_sub)
        kap_s  = self.bezier.curvature(u_sub)

        if not self.is_3d:
            speed = np.linalg.norm(d1v, axis=1, keepdims=True)
            speed = np.where(speed > 1e-10, speed, 1e-10)
            T   = d1v / speed
            N   = np.stack([-T[:, 1], T[:, 0]], axis=1)
            tip = base + scale * kap_s[:, np.newaxis] * N
            for b, t in zip(base, tip):
                self.ax.plot([b[0], t[0]], [b[1], t[1]],
                             color='#90A4AE', alpha=0.65, linewidth=1.2, zorder=2)
            self.ax.plot(tip[:, 0], tip[:, 1],
                         color='#90A4AE', alpha=0.5, linewidth=1.0,
                         linestyle='--', zorder=2, label='Curvature comb')
        else:
            cross_d1_d2 = np.cross(d1v, d2v)
            N_raw       = np.cross(cross_d1_d2, d1v)
            norm        = np.linalg.norm(N_raw, axis=1, keepdims=True)
            norm        = np.where(norm > 1e-10, norm, 1e-10)
            N           = N_raw / norm
            tip         = base + scale * kap_s[:, np.newaxis] * N
            for b, t in zip(base, tip):
                self.ax.plot([b[0], t[0]], [b[1], t[1]], [b[2], t[2]],
                             color='#90A4AE', alpha=0.65, linewidth=1.2)
            self.ax.plot(tip[:, 0], tip[:, 1], tip[:, 2],
                         color='#90A4AE', alpha=0.5, linewidth=1.0,
                         linestyle='--', label='Curvature comb')

    def _set_decorations(self):
        """Title, axis labels, legend — adapted for 2D/3D."""
        title = (
            f'Bezier Curve: {self.bezier.degree_name} '
            f'({self.bezier.dim_name}, '
            f'{self.bezier.n} Control Points)'
        )
        self.ax.set_title(
            title, fontsize=13, fontweight='bold', pad=14
        )
        self.ax.set_xlabel('x', fontsize=11)
        self.ax.set_ylabel('y', fontsize=11)
        if self.is_3d:
            self.ax.set_zlabel('z', fontsize=11)
        self.ax.legend(fontsize=8, loc='lower right')

        if not self.is_3d:
            pts   = self.bezier.points
            curve = self.bezier.curve()
            all_x = np.concatenate([pts[:, 0], curve[:, 0]])
            all_y = np.concatenate([pts[:, 1], curve[:, 1]])
            m = 0.8
            self.ax.set_xlim(all_x.min() - m, all_x.max() + m)
            self.ax.set_ylim(all_y.min() - m, all_y.max() + m)

    # -- Main ------------------------------------------------

    def draw(self):
        """Compose all plot elements."""
        self._draw_comb()
        self._draw_curve()
        self._draw_control_polygon()
        self._label_control_points()
        self._draw_highlight_points()
        self._draw_de_casteljau()
        self._add_equation_box()
        self._set_decorations()

    def save(self, filename: str):
        self.fig.tight_layout()
        self.fig.savefig(filename, dpi=150)
        print(f'Saved: {filename}')

    def show(self):
        self.fig.tight_layout()
        plt.show()

In [5]:
# ============================================================
# -- Entry Point ---------------------------------------------
# ============================================================

def main() -> int:
    data   = CASE_DATA[CASE]
    bezier = BezierCurve(data['points'])
    plot   = BezierPlot(
        bezier,
        u_highlight = data['u_highlight'],
        u_dc        = data['u_dc'],
    )
    plot.draw()
    plot.save(data['output'])
    plot.show()
    return 0    # ── Curvature comb (optional) ─────────────────────────────────
    if show_comb:
        n_pts  = 300
        u_vals_c = np.linspace(0, 1, n_pts)
        scale  = bezier.comb_scale(n_pts)
        step   = max(1, n_pts // 60)
        u_sub  = u_vals_c[::step]
        d1v    = bezier.d1(u_sub)
        d2v    = bezier.d2(u_sub)
        base   = bezier._bpoly(u_sub)
        kap_s  = bezier.curvature(u_sub)
        if not is_3d:
            speed = np.linalg.norm(d1v, axis=1, keepdims=True)
            speed = np.where(speed > 1e-10, speed, 1e-10)
            T   = d1v / speed
            N   = np.stack([-T[:, 1], T[:, 0]], axis=1)
            tip = base + scale * kap_s[:, np.newaxis] * N
            for b, t in zip(base, tip):
                ax.plot([b[0], t[0]], [b[1], t[1]],
                        color='#90A4AE', alpha=0.65, linewidth=1.2, zorder=2)
            ax.plot(tip[:, 0], tip[:, 1],
                    color='#90A4AE', alpha=0.5, linewidth=1.0,
                    linestyle='--', zorder=2, label='Curvature comb')
        else:
            cross_d1_d2 = np.cross(d1v, d2v)
            N_raw       = np.cross(cross_d1_d2, d1v)
            norm        = np.linalg.norm(N_raw, axis=1, keepdims=True)
            norm        = np.where(norm > 1e-10, norm, 1e-10)
            N           = N_raw / norm
            tip         = base + scale * kap_s[:, np.newaxis] * N
            for b, t in zip(base, tip):
                ax.plot([b[0], t[0]], [b[1], t[1]], [b[2], t[2]],
                        color='#90A4AE', alpha=0.65, linewidth=1.2)
            ax.plot(tip[:, 0], tip[:, 1], tip[:, 2],
                    color='#90A4AE', alpha=0.5, linewidth=1.0,
                    linestyle='--', label='Curvature comb')

# ============================================================
# -- Entry Point ---------------------------------------------
# ============================================================

def main() -> int:
    data   = CASE_DATA[CASE]
    bezier = BezierCurve(data['points'])
    plot   = BezierPlot(
        bezier,
        u_highlight = data['u_highlight'],
        u_dc        = data['u_dc'],
    )
    plot.draw()
    plot.save(data['output'])
    plot.show()
    return 0

In [6]:
# ============================================================
# -- Interactive Widget
# ============================================================
# Two separate DC layers:
#   1. REFERENCE lines at fixed u values (0.25, 0.50, 0.75)
#      → controlled by "Show De Casteljau lines" checkbox
#   2. SLIDER line at the current u
#      → always visible, moves with the slider
# ============================================================

import io
import ipywidgets as widgets
from IPython.display import display

COLOR_CURVE    = '#1565C0'
COLOR_POLYGON  = '#E53935'
COLOR_U_LINE   = '#FF6F00'

# Fixed colours per reference u value (match reference image)
DC_REF_COLORS  = {0.25: '#E91E8C', 0.50: '#FF6F00', 0.75: '#7B1FA2'}

# Colours for slider DC levels (one per reduction level)
DC_LVL_COLORS  = ['#F57F17', '#EC407A', '#7B1FA2', '#1B5E20']

# u values shown as fixed reference lines (2D only — 3D gets too cluttered)
DC_REF_U_VALS  = [0.25, 0.50, 0.75]


def _render_to_png(bezier, show_dc_ref, u_dc, show_comb=False) -> bytes:
    """
    Render the Bezier scene to PNG bytes.

    show_dc_ref : toggles the fixed reference DC lines
    u_dc        : position of the slider DC line (always drawn)
    """
    is_3d = bezier.dim == 3
    sns.set_theme(style='whitegrid')

    if is_3d:
        fig = plt.figure(figsize=(10, 7))
        ax  = fig.add_subplot(111, projection='3d')
    else:
        fig, ax = plt.subplots(figsize=(10, 7))

    def coords(pts):
        return (pts[:,0], pts[:,1], pts[:,2]) if is_3d else (pts[:,0], pts[:,1])
    def plot_line(pts, **kw):    ax.plot(*coords(pts), **kw)
    def scatter_pts(pts, **kw):  ax.scatter(*coords(pts), **kw)

    # ── Bezier curve ─────────────────────────────────────────
    curve_pts = bezier.curve()
    plot_line(curve_pts, color=COLOR_CURVE, linewidth=2.5,
              label='Bezier Curve', zorder=3)

    # ── Control polygon ──────────────────────────────────────
    plot_line(bezier.points, color=COLOR_POLYGON, linestyle='--',
              linewidth=1.5, marker='o', markersize=8,
              label='Control Polygon', zorder=4)

    # ── Control point labels ─────────────────────────────────
    for i, pt in enumerate(bezier.points):
        cs = ', '.join(f'{v:.4g}' for v in pt)
        dy = 0.15 if i % 2 == 0 else -0.28
        if is_3d:
            ax.text(pt[0]+0.10, pt[1]+dy, pt[2]+0.05,
                    f'P{i}({cs})', fontsize=8, color=COLOR_POLYGON, fontweight='bold')
        else:
            ax.annotate(f'P{i}({cs})', xy=(pt[0], pt[1]),
                        xytext=(pt[0]+0.10, pt[1]+dy),
                        fontsize=8.5, color=COLOR_POLYGON, fontweight='bold')

    # ── FIXED reference DC lines (checkbox-controlled) ───────
    if show_dc_ref and not is_3d:
        for u_ref in DC_REF_U_VALS:
            col    = DC_REF_COLORS[u_ref]
            levels = bezier.de_casteljau(u_ref)
            # Draw each intermediate level
            for lv, pts in enumerate(levels[1:-1], start=1):
                label = f'De Casteljau u={u_ref:.2f}' if lv == 1 else '_nolegend_'
                plot_line(pts, color=col, linewidth=1.5, marker='o',
                          markersize=6, zorder=5, label=label)
                # Annotate intermediate points (Q0, Q1 etc.)
                for j, pt in enumerate(pts):
                    ax.annotate(f'Q{j}({u_ref:.2f})',
                                xy=(pt[0], pt[1]),
                                xytext=(pt[0]+0.06, pt[1]+0.06),
                                fontsize=6.5, color=col)
            # Final point (R0)
            final = levels[-1][0]
            scatter_pts(levels[-1], color=col, marker='D', s=80, zorder=7,
                        label='_nolegend_')
            ax.annotate(f'R0({u_ref:.2f})\n({final[0]:.2f},{final[1]:.2f})',
                        xy=(final[0], final[1]),
                        xytext=(final[0]+0.08, final[1]-0.22),
                        fontsize=6.5, color=col, fontweight='bold')

    # ── SLIDER DC line (always shown) ────────────────────────
    levels_u = bezier.de_casteljau(u_dc)
    pt_u     = levels_u[-1][0]

    for lv, pts in enumerate(levels_u[1:-1], start=1):
        col   = DC_LVL_COLORS[min(lv-1, len(DC_LVL_COLORS)-1)]
        label = f'DC u={u_dc:.2f} level {lv}' if lv == 1 else '_nolegend_'
        plot_line(pts, color=col, linewidth=1.8, marker='o',
                  markersize=7, zorder=6, label=label, alpha=0.9)

    scatter_pts(levels_u[-1], color=COLOR_U_LINE, marker='D',
                s=110, zorder=8, label=f'u = {u_dc:.2f}')

    # Drop line + annotation for slider point
    if is_3d:
        z_floor = curve_pts[:,2].min()
        ax.plot([pt_u[0]]*2, [pt_u[1]]*2, [pt_u[2], z_floor],
                color=COLOR_U_LINE, linestyle=':', linewidth=1.8, zorder=6)
        ax.text(pt_u[0]+0.08, pt_u[1]+0.08, pt_u[2]+0.05,
                f'u={u_dc:.2f}', fontsize=8, color=COLOR_U_LINE, fontweight='bold')
    else:
        all_pts = np.vstack([bezier.points, curve_pts])
        y_floor = all_pts[:,1].min() - 0.8
        ax.plot([pt_u[0], pt_u[0]], [y_floor, pt_u[1]],
                color=COLOR_U_LINE, linestyle=':', linewidth=1.8, zorder=6)
        ax.annotate(f'u={u_dc:.2f}',
                    xy=(pt_u[0], pt_u[1]),
                    xytext=(pt_u[0]+0.12, pt_u[1]+0.15),
                    fontsize=9, color=COLOR_U_LINE, fontweight='bold',
                    bbox=dict(boxstyle='round,pad=0.25', fc='white', alpha=0.75, ec='none'))

    # ── Curvature comb (optional) ─────────────────────────────────
    if show_comb:
        n_pts_c = 300
        u_vals_c = np.linspace(0, 1, n_pts_c)
        scale  = bezier.comb_scale(n_pts_c)
        step   = max(1, n_pts_c // 60)
        u_sub  = u_vals_c[::step]
        d1v    = bezier.d1(u_sub)
        d2v    = bezier.d2(u_sub)
        base   = bezier._bpoly(u_sub)
        kap_s  = bezier.curvature(u_sub)
        if not is_3d:
            speed = np.linalg.norm(d1v, axis=1, keepdims=True)
            speed = np.where(speed > 1e-10, speed, 1e-10)
            T   = d1v / speed
            N   = np.stack([-T[:, 1], T[:, 0]], axis=1)
            tip = base + scale * kap_s[:, np.newaxis] * N
            for b, t in zip(base, tip):
                ax.plot([b[0], t[0]], [b[1], t[1]],
                        color='#90A4AE', alpha=0.65, linewidth=1.2, zorder=2)
            ax.plot(tip[:, 0], tip[:, 1],
                    color='#90A4AE', alpha=0.5, linewidth=1.0,
                    linestyle='--', zorder=2, label='Curvature comb')
        else:
            cross_d1_d2 = np.cross(d1v, d2v)
            N_raw       = np.cross(cross_d1_d2, d1v)
            norm        = np.linalg.norm(N_raw, axis=1, keepdims=True)
            norm        = np.where(norm > 1e-10, norm, 1e-10)
            N           = N_raw / norm
            tip         = base + scale * kap_s[:, np.newaxis] * N
            for b, t in zip(base, tip):
                ax.plot([b[0], t[0]], [b[1], t[1]], [b[2], t[2]],
                        color='#90A4AE', alpha=0.65, linewidth=1.2)
            ax.plot(tip[:, 0], tip[:, 1], tip[:, 2],
                    color='#90A4AE', alpha=0.5, linewidth=1.0,
                    linestyle='--', label='Curvature comb')

    # ── Decorations ──────────────────────────────────────────
    ax.set_title(f'Bezier Curve: {bezier.degree_name} '
                 f'({bezier.dim_name}, {bezier.n} Control Points)',
                 fontsize=13, fontweight='bold', pad=14)
    ax.set_xlabel('x', fontsize=11)
    ax.set_ylabel('y', fontsize=11)
    if is_3d:
        ax.set_zlabel('z', fontsize=11)
    if not is_3d:
        all_x = np.concatenate([bezier.points[:,0], curve_pts[:,0]])
        all_y = np.concatenate([bezier.points[:,1], curve_pts[:,1]])
        m = 0.8
        ax.set_xlim(all_x.min()-m, all_x.max()+m)
        ax.set_ylim(all_y.min()-m, all_y.max()+m)
    ax.legend(fontsize=8, loc='lower right')
    fig.tight_layout()

    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=110)
    plt.close(fig)
    buf.seek(0)
    return buf.read()


In [7]:
# ── Widgets ───────────────────────────────────────────────────
_case_labels = {
    1: 'Case 1 — Quadratic 2D',
    2: 'Case 2 — Quartic 3D',
}

case_toggle = widgets.ToggleButtons(
    options      = list(_case_labels.values()),
    value        = _case_labels[CASE],
    description  = '',
    button_style = '',
    style        = {'button_width': '180px'},
)

u_slider = widgets.FloatSlider(
    value             = 0.50,
    min               = 0.0,
    max               = 1.0,
    step              = 0.01,
    description       = 'u (slider) :',
    continuous_update = True,
    style             = {'description_width': 'initial'},
    layout            = widgets.Layout(width='520px'),
)

show_dc_check = widgets.Checkbox(
    value       = True,
    description = 'Show reference De Casteljau lines (u=0.25, 0.50, 0.75)',
    indent      = False,
)

show_comb_check = widgets.Checkbox(
    value       = False,
    description = 'Show curvature comb',
    indent      = False,
)

_img = widgets.Image(format='png',
                     layout=widgets.Layout(width='780px'))

def _get_case_num():
    return next(k for k, v in _case_labels.items() if v == case_toggle.value)

def _update(change=None):
    bezier = BezierCurve(CASE_DATA[_get_case_num()]['points'])
    _img.value = _render_to_png(bezier,
                                 show_dc_ref=show_dc_check.value,
                                 u_dc=u_slider.value,
                                 show_comb=show_comb_check.value)

case_toggle.observe(_update, names='value')
u_slider.observe(_update, names='value')
show_dc_check.observe(_update, names='value')
show_comb_check.observe(_update, names='value')

_update()
display(widgets.VBox([case_toggle, u_slider, show_dc_check, show_comb_check, _img]))
